# xG-Spark: Expected Goals with Apache Spark
## Phase 2: Ingestion (Bronze Layer)

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # this notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from src.config import BRONZE_DIR
from src.spark_session import get_spark

spark = get_spark("xG-notebook")
spark


In [2]:
competitions = spark.read.parquet((BRONZE_DIR / "competitions").as_posix())
matches = spark.read.parquet((BRONZE_DIR / "matches").as_posix())
lineups = spark.read.parquet((BRONZE_DIR / "lineups").as_posix())
events = spark.read.parquet((BRONZE_DIR / "events").as_posix())

for name, df in [("competitions", competitions), ("matches", matches),
                 ("lineups", lineups), ("events", events)]:
    print(f"{name:<13} {df.count():>10,} rows  {len(df.columns):>3} columns")


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/C:/Users/adity/Desktop/xg_spark/data/bronze/competitions.

In [ ]:
(events.groupBy("competition_id", "season_id").count()
    .join(competitions.select("competition_id", "season_id",
                              "competition_name", "season_name"),
          on=["competition_id", "season_id"])
    .orderBy(F.desc("count"))
    .show(truncate=False))


In [ ]:
(events.groupBy(F.col("type.name").alias("event_type"))
    .count()
    .orderBy(F.desc("count"))
    .show(15))


In [ ]:
events.select("shot").printSchema()


In [ ]:
(events.filter(F.col("type.name") == "Shot")
    .select("match_id", "minute",
            F.col("player.name").alias("player"),
            "location",
            F.col("shot.outcome.name").alias("outcome"),
            F.col("shot.statsbomb_xg").alias("statsbomb_xg"))
    .show(10, truncate=False))


In [ ]:
print("Partitions in events DataFrame:", events.rdd.getNumPartitions())

# Look for "PartitionFilters" in the plan: Spark reads ONLY the Premier League folder
events.filter(F.col("competition_id") == 2).select("id").explain()
